# 05 — Serving con Flask

In [ ]:
!pip install flask pydantic requests -q

In [ ]:
from flask import Flask, jsonify, request
from pydantic import BaseModel, Field, field_validator
import time
import json
import warnings
warnings.filterwarnings('ignore')
print("Imports OK")

## 1. Arquitectura del Serving con Flask

```
Cliente HTTP          Flask App                MLflow Registry
    │                     │                         │
    │  POST /predict       │                         │
    │─────────────────────▶│                         │
    │                      │   load_model()          │
    │                      │────────────────────────▶│
    │                      │◀────────────────────────│
    │                      │  model.predict(X)       │
    │                      │─────────────────────────│
    │  {"prediction": ...} │                         │
    │◀─────────────────────│                         │
```

Flask es un microframework web de Python ideal para construir APIs de inferencia ligeras.
En este notebook simulamos un servidor de predicciones completo usando `test_client`
para no tener que levantar un servidor real.

## 2. Los 4 endpoints principales

| Endpoint | Método | Descripción |
|----------|--------|-------------|
| `/health` | GET | Verifica que el servicio está activo y retorna la versión del modelo |
| `/model-info` | GET | Retorna metadata del modelo: nombre, versión, stage y métricas |
| `/predict` | POST | Recibe un registro JSON y retorna una predicción individual |
| `/predict-batch` | POST | Recibe múltiples registros y retorna predicciones en lote |

Esta estructura de endpoints es una convención común en APIs de ML en producción.
- **`/health`** es esencial para load balancers y kubernetes liveness probes.
- **`/model-info`** permite auditar qué versión del modelo está sirviendo sin acceder a logs.
- **`/predict`** es el endpoint principal para uso online (baja latencia, 1 registro).
- **`/predict-batch`** optimiza el throughput cuando se necesita predecir muchos registros a la vez.

## 3. Crear la aplicación Flask

In [ ]:
app = Flask(__name__)

# Simular un modelo cargado (mock)
class MockModel:
    def predict(self, features):
        # Simulación simple: suma ponderada de features
        return [features.get("hour", 12) * 50 + features.get("temperature_c", 25) * 3 + 800]

model = MockModel()
MODEL_VERSION = "v1.0-demo"

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "model_version": MODEL_VERSION,
        "timestamp": "2024-01-15T10:30:00Z"
    })

@app.route("/model-info", methods=["GET"])
def model_info():
    return jsonify({
        "model_name": "demand_forecaster",
        "version": MODEL_VERSION,
        "stage": "Production",
        "metrics": {"rmse": 45.3, "r2": 0.92}
    })

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    if not data:
        return jsonify({"error": "Body JSON requerido"}), 400

    # Validación básica
    required = ["hour", "temperature_c"]
    missing = [f for f in required if f not in data]
    if missing:
        return jsonify({"error": f"Campos faltantes: {missing}"}), 422

    if not (0 <= data["hour"] <= 23):
        return jsonify({"error": "hour debe estar entre 0 y 23"}), 422

    t0 = time.time()
    prediction = model.predict(data)[0]
    latency = (time.time() - t0) * 1000

    return jsonify({
        "prediction": round(prediction, 2),
        "unit": "kWh",
        "model_version": MODEL_VERSION,
        "latency_ms": round(latency, 3)
    })

@app.route("/predict-batch", methods=["POST"])
def predict_batch():
    data = request.get_json()
    records = data.get("records", [])
    if not records:
        return jsonify({"error": "records no puede estar vacío"}), 400

    predictions = [round(model.predict(r)[0], 2) for r in records]
    return jsonify({"predictions": predictions, "count": len(predictions)})

print("Flask app creada. Probando con test_client...")

## 4. Probar todos los endpoints con test_client

In [ ]:
with app.test_client() as client:
    # GET /health
    resp = client.get("/health")
    print("GET /health:", resp.get_json())

    # GET /model-info
    resp = client.get("/model-info")
    print("GET /model-info:", resp.get_json())

    # POST /predict - válido
    payload = {"hour": 19, "temperature_c": 28.5, "is_weekend": 0}
    resp = client.post("/predict", json=payload)
    print("POST /predict:", resp.get_json())

    # POST /predict - inválido (hour fuera de rango)
    payload_bad = {"hour": 25, "temperature_c": 28.5}
    resp = client.post("/predict", json=payload_bad)
    print("POST /predict (inválido):", resp.status_code, resp.get_json())

    # POST /predict-batch
    batch = {"records": [
        {"hour": 8, "temperature_c": 22},
        {"hour": 14, "temperature_c": 30},
        {"hour": 20, "temperature_c": 27},
    ]}
    resp = client.post("/predict-batch", json=batch)
    print("POST /predict-batch:", resp.get_json())

## 5. Benchmark de latencia

In [ ]:
import statistics

latencies = []
with app.test_client() as client:
    for i in range(100):
        payload = {"hour": i % 24, "temperature_c": 25 + (i % 10)}
        t0 = time.time()
        client.post("/predict", json=payload)
        latencies.append((time.time() - t0) * 1000)

print(f"100 requests completados:")
print(f"  Latencia media:   {statistics.mean(latencies):.2f} ms")
print(f"  Latencia mediana: {statistics.median(latencies):.2f} ms")
print(f"  P95:              {sorted(latencies)[94]:.2f} ms")
print(f"  P99:              {sorted(latencies)[98]:.2f} ms")

## Limitaciones de Flask y alternativas

Flask es una excelente opción para comenzar, pero tiene limitaciones para producción a escala.

| Característica | Flask | FastAPI | BentoML | Seldon |
|----------------|-------|---------|---------|--------|
| Velocidad de desarrollo | Alta | Alta | Media | Baja |
| Performance async | No nativa | Sí (nativa) | Sí | Sí |
| Validación automática | No | Sí (Pydantic) | Parcial | No |
| Documentación auto | No | Sí (OpenAPI) | Sí | Sí |
| Batching inteligente | No | No | Sí | Sí |
| Multi-model serving | Manual | Manual | Sí | Sí |
| Kubernetes nativo | No | No | Parcial | Sí |
| A/B testing | Manual | Manual | Sí | Sí |
| Mejor para | Prototipos, APIs simples | APIs REST modernas | ML serving | Producción enterprise |

**Recomendación:**
- **Flask**: prototipado rápido, proyectos pequeños, cuando ya conoces Flask.
- **FastAPI**: nueva API REST con validación y docs automáticos — reemplaza a Flask en la mayoría de casos.
- **BentoML**: cuando necesitas features específicas de ML serving (batching, model store).
- **Seldon**: producción enterprise en Kubernetes con A/B testing y canary deployments.

## Resumen

En este notebook aprendimos:

1. **Arquitectura**: cómo un cliente HTTP interactúa con una Flask App que carga un modelo de MLflow Registry.
2. **Endpoints estándar**: `/health`, `/model-info`, `/predict`, `/predict-batch` — convención común en APIs de ML.
3. **Validación de inputs**: campos requeridos, rangos válidos, errores HTTP correctos (400, 422).
4. **Testing sin servidor**: `app.test_client()` permite probar todos los endpoints en un notebook.
5. **Latencia**: el benchmark mostró la latencia de inferencia bajo carga local.
6. **Alternativas**: Flask es ideal para prototipos, pero FastAPI/BentoML/Seldon son preferibles en producción.

**Próximo notebook**: `06_serving_mlflow.ipynb` — serving con `mlflow models serve` y despliegue en SageMaker.